# Prototype-Conditioned Wavelet Residual Adapter (PCWRA)

This notebook implements a **new, more stable idea** after your failed:
- PBT search
- DBLoss full run
- adaptive lookback full run
- gated residual pilot
- HeRSMix pilot

## Research-grounded idea

Recent work suggests three important lessons:

1. **No model is a universal champion** in LTSF, so forcing one inductive bias everywhere is often wrong.  
2. **Pattern-specific experts under distribution shift** can help more than one global model.  
3. **Selective learning** and **patch-wise structural losses** can improve generalization by avoiding overfitting to noisy or structurally mismatched targets.

This notebook combines those lessons into one concrete model:

### Model
- **Base predictor**: simple target-channel DLinear-style predictor
- **Prototype-conditioned residual experts**: K residual heads specialized to pattern prototypes
- **Prototype features**: trend / variance / autocorrelation / FFT high-frequency ratio / Haar-wavelet energy ratios
- **Soft prototype routing**: assignment by distance to prototype centroids from training windows
- **Residual input**: raw target history + Haar multiscale coefficients
- **Loss**:
  - point-wise MSE
  - patch-wise structural loss
  - selective weighting using routing entropy
  - light residual specialization regularizer

## Strongest defensible claim
A **prototype-conditioned wavelet residual adapter** can handle patch-level distribution shift more stably than learned routers by assigning residual correction through offline-discovered regime prototypes.

## Weakest claim you should NOT make
Do **not** claim universal state-of-the-art across all 9 datasets unless the full benchmark truly proves it.

## Minimal kill test
If `pcwra` does **not** beat both:
- `base_only`
- `fixed_global_residual`

on at least **3/4 pilot datasets**, reject the idea.

In [2]:
# ============================================
# CONFIG
# ============================================

RUN_MODE = "pilot_then_full"   # options: "pilot_only", "pilot_then_full", "full_only"
AUTO_FULL_IF_PILOT_GOOD = True

PILOT_DATASETS = ["etth1", "weather", "traffic", "ili"]

# pilot success rule:
# 1) pcwra beats base_only on >= 3 pilot datasets
# 2) pcwra beats fixed_global_residual on >= 3 pilot datasets
PILOT_THRESH_BASE = 3
PILOT_THRESH_GLOBAL = 3

SEED = 42
ROOT_DIR = "/data/Sajjan_Singh/spml/wavelet_seq_project"

In [3]:
# ============================================
# IMPORTS AND SETUP
# ============================================

from pathlib import Path
import sys
import importlib
import gc
import json
import random
import copy
import time
import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

ROOT = Path(ROOT_DIR).resolve()
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

OUT_TABLES = ROOT / "results" / "tables" / "phase12_pcwra"
OUT_PREDS = ROOT / "results" / "predictions" / "phase12_pcwra"
OUT_CKPTS = ROOT / "results" / "checkpoints" / "phase12_pcwra"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_PREDS.mkdir(parents=True, exist_ok=True)
OUT_CKPTS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = ROOT / "results" / "tables" / "master_all_models_metrics_unified.csv"
BEST_PATH = ROOT / "results" / "tables" / "master_best_by_dataset_unified.csv"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
else:
    GPU_NAME = "CPU"
    GPU_MEM_GB = 0.0

USE_AMP = bool(torch.cuda.is_available()) and bool(getattr(p5, "TRAIN_CFG", {}).get("use_amp", True))
PATIENCE = int(getattr(p5, "TRAIN_CFG", {}).get("full_patience", 8))

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE:", p5.DEVICE)
print("GPU_NAME:", GPU_NAME)
print("GPU_MEM_GB:", round(GPU_MEM_GB, 2))
print("USE_AMP:", USE_AMP)
print("PATIENCE:", PATIENCE)
print("All datasets:", p5.ALL_DATASETS)

Imported p5 from: /data/Sajjan_Singh/spml/wavelet_seq_project/scripts/phase5_adaptive_wavelet_model.py
DEVICE: cuda
GPU_NAME: NVIDIA H100 NVL
GPU_MEM_GB: 93.0
USE_AMP: True
PATIENCE: 5
All datasets: ['etth1', 'etth2', 'ettm1', 'ettm2', 'weather', 'electricity', 'traffic', 'exchange', 'ili']


In [4]:
# ============================================
# FIXED SETTINGS
# ============================================

ALL_DATASETS = list(p5.ALL_DATASETS)

if GPU_MEM_GB >= 80:
    BATCH_MUL = {
        "etth1": 2, "etth2": 2, "ettm1": 2, "ettm2": 2,
        "weather": 2, "electricity": 1, "traffic": 1,
        "exchange": 2, "ili": 2,
    }
else:
    BATCH_MUL = {ds: 1 for ds in ALL_DATASETS}

PCWRA_CFG = {
    "num_prototypes": 4,
    "prototype_temperature": 0.25,
    "resid_hidden": 128,
    "lambda_ps": 0.25,
    "lambda_selective": 1.0,
    "lambda_resid_aux": 0.10,
    "selector_keep_ratio": 0.75,
    "patch_size": 12,
    "global_resid_hidden": 128,
}

print("PCWRA_CFG:", PCWRA_CFG)
print("BATCH_MUL:", BATCH_MUL)

PCWRA_CFG: {'num_prototypes': 4, 'prototype_temperature': 0.25, 'resid_hidden': 128, 'lambda_ps': 0.25, 'lambda_selective': 1.0, 'lambda_resid_aux': 0.1, 'selector_keep_ratio': 0.75, 'patch_size': 12, 'global_resid_hidden': 128}
BATCH_MUL: {'etth1': 2, 'etth2': 2, 'ettm1': 2, 'ettm2': 2, 'weather': 2, 'electricity': 1, 'traffic': 1, 'exchange': 2, 'ili': 2}


In [5]:
# ============================================
# HELPERS
# ============================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))

def resolve_seq_len(ds):
    seq_candidates = p5.resolve_seq_candidates(ds, "searched")
    return int(seq_candidates[min(1, len(seq_candidates) - 1)])

def resolve_batch_size(ds):
    return int(p5.DEFAULT_BATCH_SIZE[ds] * BATCH_MUL[ds])

def resolve_epochs(ds):
    return int(p5.DEFAULT_EPOCHS[ds])

def metric_row_from_npz(dataset, model, family, seq_len, pred_len, pred_file_rel):
    p = ROOT / str(pred_file_rel)
    arr = np.load(p, allow_pickle=True)
    preds = np.asarray(arr["preds"], dtype=np.float64).reshape(-1)
    trues = np.asarray(arr["trues"], dtype=np.float64).reshape(-1)
    return {
        "dataset": dataset,
        "model": model,
        "family": family,
        "seq_len": int(seq_len),
        "pred_len": int(pred_len),
        "test_mse": mse(trues, preds),
        "test_mae": mae(trues, preds),
        "test_rmse": rmse(trues, preds),
        "prediction_file": str(pred_file_rel),
    }

def moving_avg_1d(x, kernel):
    if kernel <= 1:
        return x
    pad = kernel // 2
    x3 = x.unsqueeze(1)
    xpad = F.pad(x3, (pad, pad), mode="replicate")
    out = F.avg_pool1d(xpad, kernel_size=kernel, stride=1)
    return out.squeeze(1)

def patchify(x, patch_size):
    # x: [B,H]
    B, H = x.shape
    n = H // patch_size
    x = x[:, :n * patch_size]
    return x.reshape(B, n, patch_size)

def patchwise_structural_loss(pred, true, patch_size=12):
    pred_p = patchify(pred, patch_size)
    true_p = patchify(true, patch_size)

    mu_p = pred_p.mean(dim=-1)
    mu_t = true_p.mean(dim=-1)

    var_p = pred_p.var(dim=-1, unbiased=False)
    var_t = true_p.var(dim=-1, unbiased=False)

    p_center = pred_p - mu_p.unsqueeze(-1)
    t_center = true_p - mu_t.unsqueeze(-1)
    corr_num = (p_center * t_center).mean(dim=-1)
    corr_den = p_center.pow(2).mean(dim=-1).sqrt() * t_center.pow(2).mean(dim=-1).sqrt() + 1e-6
    corr = corr_num / corr_den

    loss = (
        F.mse_loss(mu_p, mu_t)
        + F.mse_loss(var_p, var_t)
        + ((1.0 - corr) ** 2).mean()
    )
    return loss

def haar_multiscale_features(x):
    '''
    x: [B,T]
    returns concatenated multiscale approx/detail summaries
    '''
    feats = [x]
    cur = x
    for _ in range(3):
        if cur.shape[1] < 2:
            break
        even = cur[:, 0::2]
        odd = cur[:, 1::2]
        m = min(even.shape[1], odd.shape[1])
        even = even[:, :m]
        odd = odd[:, :m]
        approx = (even + odd) / 2.0
        detail = (even - odd) / 2.0
        feats.append(approx)
        feats.append(detail)
        cur = approx
    out = []
    for f in feats:
        out.append(f.mean(dim=1, keepdim=True))
        out.append(f.std(dim=1, keepdim=True))
        out.append(f.abs().mean(dim=1, keepdim=True))
        out.append((f ** 2).mean(dim=1, keepdim=True))
    return torch.cat(out, dim=1)

def build_prototype_features_np(x_np):
    '''
    x_np: [N,T] target-channel windows on CPU numpy
    Returns feature matrix for offline clustering
    '''
    x = torch.tensor(x_np, dtype=torch.float32)
    spec = torch.fft.rfft(x, dim=1)
    power = spec.real.pow(2) + spec.imag.pow(2)
    total_power = power.sum(dim=1) + 1e-8
    Fbins = power.shape[1]
    split = max(1, int(round(Fbins * 0.50)))
    high_ratio = power[:, split:].sum(dim=1) / total_power

    mean_abs = x.abs().mean(dim=1)
    std = x.std(dim=1)
    if x.shape[1] >= 2:
        slope = x[:, -1] - x[:, 0]
    else:
        slope = torch.zeros_like(mean_abs)

    x1 = x[:, 1:]
    x0 = x[:, :-1]
    num = (x1 * x0).mean(dim=1)
    den = x1.pow(2).mean(dim=1).sqrt() * x0.pow(2).mean(dim=1).sqrt() + 1e-6
    lag1 = num / den

    wave = haar_multiscale_features(x)

    feats = torch.cat([
        mean_abs.unsqueeze(1),
        std.unsqueeze(1),
        slope.unsqueeze(1),
        high_ratio.unsqueeze(1),
        lag1.unsqueeze(1),
        wave
    ], dim=1)

    return feats.numpy().astype(np.float32)

def select_easy_mask(score, keep_ratio=0.75):
    B = score.shape[0]
    k = max(1, int(round(B * keep_ratio)))
    thresh = torch.kthvalue(score.detach(), k).values
    mask = (score <= thresh).float()
    return mask

In [6]:
# ============================================
# MODEL COMPONENTS
# ============================================

class DLinearTargetOnly(nn.Module):
    def __init__(self, seq_len, pred_len, kernel_size=25):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.kernel_size = int(kernel_size)
        self.trend_linear = nn.Linear(self.seq_len, self.pred_len)
        self.seasonal_linear = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x_target):
        trend = moving_avg_1d(x_target, self.kernel_size)
        seasonal = x_target - trend
        return self.trend_linear(trend) + self.seasonal_linear(seasonal)

class GlobalResidualMLP(nn.Module):
    '''
    Stronger fixed residual baseline than learned routing.
    '''
    def __init__(self, seq_len, pred_len, hidden=128):
        super().__init__()
        in_dim = seq_len + 24  # raw target history + Haar stats
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, pred_len),
        )

    def forward(self, x_target):
        wave = haar_multiscale_features(x_target)
        feat = torch.cat([x_target, wave], dim=1)
        return self.net(feat)

class PrototypeResidualBank(nn.Module):
    '''
    K residual experts specialized by offline prototypes.
    '''
    def __init__(self, seq_len, pred_len, num_prototypes=4, hidden=128):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.num_prototypes = int(num_prototypes)
        in_dim = seq_len + 24
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.GELU(),
                nn.Linear(hidden, hidden),
                nn.GELU(),
                nn.Linear(hidden, pred_len),
            )
            for _ in range(self.num_prototypes)
        ])
        self.register_buffer("prototype_centers", torch.zeros(self.num_prototypes, 29))  # to be overwritten

    def set_prototypes(self, centers_np):
        centers = torch.tensor(centers_np, dtype=torch.float32)
        self.prototype_centers = centers

    def routing_from_features(self, proto_feat):
        # proto_feat: [B,D]
        dist = torch.cdist(proto_feat, self.prototype_centers.to(proto_feat.device), p=2) ** 2
        weights = torch.softmax(-dist / PCWRA_CFG["prototype_temperature"], dim=1)
        return weights, dist

    def forward(self, x_target, proto_feat):
        wave = haar_multiscale_features(x_target)
        feat = torch.cat([x_target, wave], dim=1)

        weights, dist = self.routing_from_features(proto_feat)

        expert_outs = []
        for head in self.heads:
            expert_outs.append(head(feat))
        expert_outs = torch.stack(expert_outs, dim=1)  # [B,K,H]

        resid = (weights.unsqueeze(-1) * expert_outs).sum(dim=1)
        return resid, weights, dist, expert_outs

class PCWRA(nn.Module):
    '''
    Modes:
    - base_only
    - fixed_global_residual
    - pcwra
    '''
    def __init__(self, seq_len, pred_len, target_idx, num_prototypes=4, mode="pcwra"):
        super().__init__()
        assert mode in ["base_only", "fixed_global_residual", "pcwra"]
        self.mode = mode
        self.target_idx = int(target_idx)

        self.base = DLinearTargetOnly(seq_len=seq_len, pred_len=pred_len, kernel_size=25)
        self.global_resid = GlobalResidualMLP(seq_len=seq_len, pred_len=pred_len, hidden=PCWRA_CFG["global_resid_hidden"])
        self.proto_bank = PrototypeResidualBank(
            seq_len=seq_len,
            pred_len=pred_len,
            num_prototypes=num_prototypes,
            hidden=PCWRA_CFG["resid_hidden"],
        )

    def set_prototypes(self, centers_np):
        self.proto_bank.set_prototypes(centers_np)

    def forward(self, x_scaled):
        x_target = x_scaled[:, :, self.target_idx]
        base_pred = self.base(x_target)

        if self.mode == "base_only":
            B = x_target.shape[0]
            zeros = torch.zeros_like(base_pred)
            proto_feat = torch.tensor(build_prototype_features_np(x_target.detach().cpu().numpy()),
                                      dtype=torch.float32, device=x_target.device)
            aux = {
                "base_pred": base_pred,
                "resid_pred": zeros,
                "proto_weights": torch.zeros(B, self.proto_bank.num_prototypes, device=x_target.device),
                "proto_entropy": torch.zeros(B, device=x_target.device),
                "proto_feat": proto_feat,
                "expert_outs": None,
            }
            return base_pred, aux

        if self.mode == "fixed_global_residual":
            resid = self.global_resid(x_target)
            proto_feat = torch.tensor(build_prototype_features_np(x_target.detach().cpu().numpy()),
                                      dtype=torch.float32, device=x_target.device)
            B = x_target.shape[0]
            aux = {
                "base_pred": base_pred,
                "resid_pred": resid,
                "proto_weights": torch.zeros(B, self.proto_bank.num_prototypes, device=x_target.device),
                "proto_entropy": torch.zeros(B, device=x_target.device),
                "proto_feat": proto_feat,
                "expert_outs": None,
            }
            return base_pred + resid, aux

        # pcwra
        proto_feat = torch.tensor(build_prototype_features_np(x_target.detach().cpu().numpy()),
                                  dtype=torch.float32, device=x_target.device)
        resid, weights, dist, expert_outs = self.proto_bank(x_target, proto_feat)
        proto_entropy = -(weights.clamp_min(1e-8).log() * weights).sum(dim=1)

        aux = {
            "base_pred": base_pred,
            "resid_pred": resid,
            "proto_weights": weights,
            "proto_entropy": proto_entropy,
            "proto_feat": proto_feat,
            "expert_outs": expert_outs,
        }
        return base_pred + resid, aux

def pcwra_loss(pred_scaled, y_scaled, aux, mode):
    point_mse_per = ((pred_scaled - y_scaled) ** 2).mean(dim=1)

    if mode == "base_only":
        base = point_mse_per.mean()
        return base, {
            "point_mse": float(base.detach().cpu().item()),
            "ps_loss": 0.0,
            "selective_keep": 1.0,
            "entropy_reg": 0.0,
            "resid_aux": 0.0,
        }

    ps = patchwise_structural_loss(pred_scaled, y_scaled, patch_size=PCWRA_CFG["patch_size"])

    base_pred = aux["base_pred"]
    resid_pred = aux["resid_pred"]
    resid_target = (y_scaled - base_pred).detach()
    resid_aux = F.mse_loss(resid_pred, resid_target)

    if mode == "fixed_global_residual":
        point = point_mse_per.mean()
        total = point + PCWRA_CFG["lambda_ps"] * ps + PCWRA_CFG["lambda_resid_aux"] * resid_aux
        return total, {
            "point_mse": float(point.detach().cpu().item()),
            "ps_loss": float(ps.detach().cpu().item()),
            "selective_keep": 1.0,
            "entropy_reg": 0.0,
            "resid_aux": float(resid_aux.detach().cpu().item()),
        }

    # pcwra
    proto_entropy = aux["proto_entropy"]
    easy_mask = select_easy_mask(proto_entropy, keep_ratio=PCWRA_CFG["selector_keep_ratio"])
    selective_point = (point_mse_per * easy_mask).sum() / (easy_mask.sum() + 1e-6)
    entropy_reg = proto_entropy.mean()

    total = (
        PCWRA_CFG["lambda_selective"] * selective_point
        + PCWRA_CFG["lambda_ps"] * ps
        + PCWRA_CFG["lambda_resid_aux"] * resid_aux
        + 1e-3 * entropy_reg
    )
    return total, {
        "point_mse": float(selective_point.detach().cpu().item()),
        "ps_loss": float(ps.detach().cpu().item()),
        "selective_keep": float(easy_mask.mean().detach().cpu().item()),
        "entropy_reg": float(entropy_reg.detach().cpu().item()),
        "resid_aux": float(resid_aux.detach().cpu().item()),
    }

In [9]:
# ============================================
# PROTOTYPE FITTING + TRAIN/EVAL  (CORRECTED)
# ============================================

def resolve_train_end_from_bundle_or_disk(dataset_name, bundle):
    """
    Robustly find train_end.
    Tries bundle keys first, then split json on disk, then known fallback counts.
    """

    # 1) direct bundle cases
    if isinstance(bundle, dict):
        # common direct keys
        for k in ["train_end", "train_rows", "n_train", "train_size"]:
            if k in bundle:
                return int(bundle[k])

        # nested split indices
        if "split_indices" in bundle and isinstance(bundle["split_indices"], dict):
            si = bundle["split_indices"]
            for k in ["train_end", "train_rows", "n_train", "train_size"]:
                if k in si:
                    return int(si[k])

            if "train" in si:
                tr = si["train"]
                if isinstance(tr, int):
                    return int(tr)
                if isinstance(tr, (list, tuple)) and len(tr) == 2:
                    start, end = int(tr[0]), int(tr[1])
                    return end + 1 if end >= start else int(tr[1])
                if isinstance(tr, dict):
                    for k in ["end", "stop", "rows", "n", "size"]:
                        if k in tr:
                            val = int(tr[k])
                            if k == "end" and "start" in tr:
                                return val + 1
                            return val

    # 2) split json from disk
    split_json = ROOT / "data" / "splits" / f"{dataset_name}_splits.json"
    if split_json.exists():
        try:
            data = json.loads(split_json.read_text())

            # direct keys
            for k in ["train_end", "train_rows", "n_train", "train_size"]:
                if k in data:
                    return int(data[k])

            # train field
            if "train" in data:
                tr = data["train"]
                if isinstance(tr, int):
                    return int(tr)
                if isinstance(tr, (list, tuple)) and len(tr) == 2:
                    start, end = int(tr[0]), int(tr[1])
                    return end + 1 if end >= start else int(tr[1])
                if isinstance(tr, dict):
                    for k in ["end", "stop", "rows", "n", "size"]:
                        if k in tr:
                            val = int(tr[k])
                            if k == "end" and "start" in tr:
                                return val + 1
                            return val

            # nested split_indices in file
            if "split_indices" in data and isinstance(data["split_indices"], dict):
                si = data["split_indices"]
                for k in ["train_end", "train_rows", "n_train", "train_size"]:
                    if k in si:
                        return int(si[k])

                if "train" in si:
                    tr = si["train"]
                    if isinstance(tr, int):
                        return int(tr)
                    if isinstance(tr, (list, tuple)) and len(tr) == 2:
                        start, end = int(tr[0]), int(tr[1])
                        return end + 1 if end >= start else int(tr[1])
                    if isinstance(tr, dict):
                        for k in ["end", "stop", "rows", "n", "size"]:
                            if k in tr:
                                val = int(tr[k])
                                if k == "end" and "start" in tr:
                                    return val + 1
                                return val
        except Exception as e:
            print(f"[WARN] Could not parse split json for {dataset_name}: {e}")

    # 3) final fallback using your known preparation summary
    fallback_train_rows = {
        "etth1": 8640,
        "etth2": 8640,
        "ettm1": 34560,
        "ettm2": 34560,
        "weather": 36886,
        "electricity": 18412,
        "traffic": 12280,
        "exchange": 5311,
        "ili": 676,
    }
    if dataset_name in fallback_train_rows:
        return int(fallback_train_rows[dataset_name])

    raise RuntimeError(
        f"Could not determine train_end for dataset={dataset_name}. "
        f"Available bundle keys: {list(bundle.keys()) if isinstance(bundle, dict) else 'not-a-dict'}"
    )


def fit_prototypes_for_dataset(dataset_name, bundle, seq_len, num_prototypes=4, max_samples=4000, seed=42):
    values = bundle["values_scaled"]
    target_idx = bundle["target_idx"]

    train_end = resolve_train_end_from_bundle_or_disk(dataset_name, bundle)
    x = values[:train_end, target_idx]

    # build sliding windows
    windows = []
    max_start = max(1, len(x) - seq_len)
    step = max(1, max_start // max_samples)

    for s in range(0, max_start, step):
        w = x[s:s + seq_len]
        if len(w) == seq_len:
            windows.append(w)
        if len(windows) >= max_samples:
            break

    if len(windows) == 0:
        raise RuntimeError(f"No prototype windows could be built for dataset={dataset_name}")

    windows = np.stack(windows, axis=0).astype(np.float32)
    feats = build_prototype_features_np(windows)

    # in very small-data edge cases
    k = min(num_prototypes, len(feats))
    if k < 2:
        k = 1

    km = KMeans(n_clusters=k, random_state=seed, n_init=10)
    km.fit(feats)
    centers = km.cluster_centers_.astype(np.float32)

    # if k < requested prototypes, pad by repeating last center
    if k < num_prototypes:
        pad = np.repeat(centers[-1:, :], num_prototypes - k, axis=0)
        centers = np.concatenate([centers, pad], axis=0)

    return centers.astype(np.float32), feats


@torch.no_grad()
def evaluate_model(model, loader, target_mean, target_std):
    model.eval()
    preds = []
    trues = []
    entropy_rows = []
    weight_rows = []

    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_scaled, aux = model(x)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
        entropy_rows.append(aux["proto_entropy"].detach().cpu().numpy())
        weight_rows.append(aux["proto_weights"].detach().cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    ent = np.concatenate(entropy_rows, axis=0)
    weights = np.concatenate(weight_rows, axis=0)

    if weights.size > 0:
        avg_weights = weights.mean(axis=0)
    else:
        avg_weights = np.zeros(PCWRA_CFG["num_prototypes"], dtype=np.float32)

    return {
        "preds": preds,
        "trues": trues,
        "mse": mse(trues.reshape(-1), preds.reshape(-1)),
        "mae": mae(trues.reshape(-1), preds.reshape(-1)),
        "rmse": rmse(trues.reshape(-1), preds.reshape(-1)),
        "avg_proto_entropy": float(ent.mean()) if ent.size > 0 else 0.0,
        "avg_proto_weights": avg_weights,
    }


def train_one_run(dataset_name, mode):
    set_seed(SEED)

    bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
    pred_len = p5.resolve_pred_len(dataset_name, "long")
    seq_len = resolve_seq_len(dataset_name)
    batch_size = resolve_batch_size(dataset_name)
    epochs = resolve_epochs(dataset_name)

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    # IMPORTANT: only fit prototypes for the actual PCWRA mode
    centers = None
    proto_feats = None
    if mode == "pcwra":
        centers, proto_feats = fit_prototypes_for_dataset(
            dataset_name=dataset_name,
            bundle=bundle,
            seq_len=seq_len,
            num_prototypes=PCWRA_CFG["num_prototypes"],
            max_samples=4000,
            seed=SEED,
        )

    model = PCWRA(
        seq_len=seq_len,
        pred_len=pred_len,
        target_idx=int(target_idx),
        num_prototypes=PCWRA_CFG["num_prototypes"],
        mode=mode,
    ).to(p5.DEVICE)

    if mode == "pcwra":
        model.set_prototypes(centers)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
        foreach=False,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_rmse = float("inf")
    best_epoch = -1
    best_state = None
    patience_left = PATIENCE

    history = []
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()

        loss_vals = []
        point_vals = []
        ps_vals = []
        keep_vals = []
        entropy_vals = []
        resid_aux_vals = []

        for batch in train_loader:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred_scaled, aux = model(x)
                loss, parts = pcwra_loss(pred_scaled, y, aux, mode)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_vals.append(float(loss.detach().cpu().item()))
            point_vals.append(parts["point_mse"])
            ps_vals.append(parts["ps_loss"])
            keep_vals.append(parts["selective_keep"])
            entropy_vals.append(parts["entropy_reg"])
            resid_aux_vals.append(parts["resid_aux"])

        val_eval = evaluate_model(model, val_loader, target_mean, target_std)
        val_rmse = float(val_eval["rmse"])

        history.append({
            "dataset": dataset_name,
            "mode": mode,
            "epoch": epoch,
            "train_loss": float(np.mean(loss_vals)),
            "train_point_mse": float(np.mean(point_vals)),
            "train_ps_loss": float(np.mean(ps_vals)),
            "train_selective_keep": float(np.mean(keep_vals)),
            "train_entropy_reg": float(np.mean(entropy_vals)),
            "train_resid_aux": float(np.mean(resid_aux_vals)),
            "val_rmse": val_rmse,
            "val_mae": float(val_eval["mae"]),
            "val_mse": float(val_eval["mse"]),
            "val_avg_proto_entropy": float(val_eval["avg_proto_entropy"]),
            "val_avg_proto_weights_json": json.dumps(val_eval["avg_proto_weights"].tolist()),
        })

        print(
            f"[{dataset_name} | {mode}] epoch {epoch:03d} | "
            f"train_loss={np.mean(loss_vals):.6f} | val_rmse={val_rmse:.6f} | "
            f"proto_entropy={val_eval['avg_proto_entropy']:.4f} | batch={batch_size}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{dataset_name} | {mode}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    test_eval = evaluate_model(model, test_loader, target_mean, target_std)

    ckpt_path = OUT_CKPTS / f"{dataset_name}_{mode}.pt"
    pred_path = OUT_PREDS / f"{dataset_name}_{mode}_test_predictions.npz"

    save_obj = {
        "dataset": dataset_name,
        "mode": mode,
        "pcwra_cfg": PCWRA_CFG,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }
    if centers is not None:
        save_obj["prototype_centers"] = centers

    torch.save(save_obj, ckpt_path)

    np.savez_compressed(
        pred_path,
        preds=test_eval["preds"],
        trues=test_eval["trues"],
        seq_len=seq_len,
        pred_len=pred_len,
        avg_proto_entropy=np.array([test_eval["avg_proto_entropy"]], dtype=np.float32),
        avg_proto_weights=np.array(test_eval["avg_proto_weights"], dtype=np.float32),
    )

    row = {
        "dataset": dataset_name,
        "mode": mode,
        "model": f"PCWRA_{mode}",
        "family": "wavelet_prototype_residual",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "batch_size": batch_size,
        "epochs_target": epochs,
        "best_epoch": int(best_epoch),
        "train_seconds": float(time.time() - t0),
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(test_eval["mse"]),
        "test_mae": float(test_eval["mae"]),
        "test_rmse": float(test_eval["rmse"]),
        "avg_proto_entropy": float(test_eval["avg_proto_entropy"]),
        "avg_proto_weights_json": json.dumps(test_eval["avg_proto_weights"].tolist()),
    }

    hist_df = pd.DataFrame(history)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, hist_df

In [ ]:
# ============================================
# PATCH CELL FOR PCWRA NOTEBOOK
# Run this AFTER your existing cells, then rerun the PILOT RUN cell
# ============================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import gc
import copy
import time
from sklearn.cluster import KMeans

# -------------------------------------------------
# Robust train_end resolver
# -------------------------------------------------
def resolve_train_end_from_bundle_or_disk(dataset_name, bundle):
    if isinstance(bundle, dict):
        for k in ["train_end", "train_rows", "n_train", "train_size"]:
            if k in bundle:
                return int(bundle[k])

        if "split_indices" in bundle and isinstance(bundle["split_indices"], dict):
            si = bundle["split_indices"]
            for k in ["train_end", "train_rows", "n_train", "train_size"]:
                if k in si:
                    return int(si[k])

            if "train" in si:
                tr = si["train"]
                if isinstance(tr, int):
                    return int(tr)
                if isinstance(tr, (list, tuple)) and len(tr) == 2:
                    start, end = int(tr[0]), int(tr[1])
                    return end + 1 if end >= start else int(tr[1])
                if isinstance(tr, dict):
                    for kk in ["end", "stop", "rows", "n", "size"]:
                        if kk in tr:
                            val = int(tr[kk])
                            if kk == "end" and "start" in tr:
                                return val + 1
                            return val

    split_json = ROOT / "data" / "splits" / f"{dataset_name}_splits.json"
    if split_json.exists():
        try:
            data = json.loads(split_json.read_text())

            for k in ["train_end", "train_rows", "n_train", "train_size"]:
                if k in data:
                    return int(data[k])

            if "train" in data:
                tr = data["train"]
                if isinstance(tr, int):
                    return int(tr)
                if isinstance(tr, (list, tuple)) and len(tr) == 2:
                    start, end = int(tr[0]), int(tr[1])
                    return end + 1 if end >= start else int(tr[1])
                if isinstance(tr, dict):
                    for kk in ["end", "stop", "rows", "n", "size"]:
                        if kk in tr:
                            val = int(tr[kk])
                            if kk == "end" and "start" in tr:
                                return val + 1
                            return val

            if "split_indices" in data and isinstance(data["split_indices"], dict):
                si = data["split_indices"]
                for k in ["train_end", "train_rows", "n_train", "train_size"]:
                    if k in si:
                        return int(si[k])

                if "train" in si:
                    tr = si["train"]
                    if isinstance(tr, int):
                        return int(tr)
                    if isinstance(tr, (list, tuple)) and len(tr) == 2:
                        start, end = int(tr[0]), int(tr[1])
                        return end + 1 if end >= start else int(tr[1])
                    if isinstance(tr, dict):
                        for kk in ["end", "stop", "rows", "n", "size"]:
                            if kk in tr:
                                val = int(tr[kk])
                                if kk == "end" and "start" in tr:
                                    return val + 1
                                return val
        except Exception as e:
            print(f"[WARN] Could not parse split json for {dataset_name}: {e}")

    fallback_train_rows = {
        "etth1": 8640,
        "etth2": 8640,
        "ettm1": 34560,
        "ettm2": 34560,
        "weather": 36886,
        "electricity": 18412,
        "traffic": 12280,
        "exchange": 5311,
        "ili": 676,
    }
    if dataset_name in fallback_train_rows:
        return int(fallback_train_rows[dataset_name])

    raise RuntimeError(
        f"Could not determine train_end for dataset={dataset_name}. "
        f"Available bundle keys: {list(bundle.keys()) if isinstance(bundle, dict) else 'not-a-dict'}"
    )

# -------------------------------------------------
# Fixed feature functions (torch-native, efficient)
# -------------------------------------------------
def haar_multiscale_features(x):
    """
    x: [B, T]
    Returns fixed-size multiscale summary.
    For 3 levels, this yields 7 series * 4 stats = 28 dims.
    """
    feats = [x]
    cur = x
    for _ in range(3):
        if cur.shape[1] < 2:
            break
        even = cur[:, 0::2]
        odd = cur[:, 1::2]
        m = min(even.shape[1], odd.shape[1])
        even = even[:, :m]
        odd = odd[:, :m]
        approx = (even + odd) / 2.0
        detail = (even - odd) / 2.0
        feats.append(approx)
        feats.append(detail)
        cur = approx

    out = []
    for f in feats:
        out.append(f.mean(dim=1, keepdim=True))
        out.append(f.std(dim=1, keepdim=True))
        out.append(f.abs().mean(dim=1, keepdim=True))
        out.append((f ** 2).mean(dim=1, keepdim=True))
    return torch.cat(out, dim=1)

def build_prototype_features_torch(x):
    """
    x: [B, T]
    Returns torch features for routing/prototype assignment.
    """
    spec = torch.fft.rfft(x, dim=1)
    power = spec.real.pow(2) + spec.imag.pow(2)
    total_power = power.sum(dim=1) + 1e-8
    Fbins = power.shape[1]
    split = max(1, int(round(Fbins * 0.50)))
    high_ratio = power[:, split:].sum(dim=1) / total_power

    mean_abs = x.abs().mean(dim=1)
    std = x.std(dim=1)
    slope = x[:, -1] - x[:, 0] if x.shape[1] >= 2 else torch.zeros_like(mean_abs)

    if x.shape[1] >= 2:
        x1 = x[:, 1:]
        x0 = x[:, :-1]
        num = (x1 * x0).mean(dim=1)
        den = x1.pow(2).mean(dim=1).sqrt() * x0.pow(2).mean(dim=1).sqrt() + 1e-6
        lag1 = num / den
    else:
        lag1 = torch.zeros_like(mean_abs)

    wave = haar_multiscale_features(x)

    feats = torch.cat([
        mean_abs.unsqueeze(1),
        std.unsqueeze(1),
        slope.unsqueeze(1),
        high_ratio.unsqueeze(1),
        lag1.unsqueeze(1),
        wave,
    ], dim=1)
    return feats

def build_prototype_features_np(x_np):
    x = torch.tensor(x_np, dtype=torch.float32)
    feats = build_prototype_features_torch(x)
    return feats.cpu().numpy().astype(np.float32)

# -------------------------------------------------
# Fixed models (dynamic dimensions, no hardcoding)
# -------------------------------------------------
class DLinearTargetOnly(nn.Module):
    def __init__(self, seq_len, pred_len, kernel_size=25):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.kernel_size = int(kernel_size)
        self.trend_linear = nn.Linear(self.seq_len, self.pred_len)
        self.seasonal_linear = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x_target):
        trend = moving_avg_1d(x_target, self.kernel_size)
        seasonal = x_target - trend
        return self.trend_linear(trend) + self.seasonal_linear(seasonal)

class GlobalResidualMLP(nn.Module):
    """
    Stronger fixed residual baseline than learned routing.
    """
    def __init__(self, seq_len, pred_len, hidden=128):
        super().__init__()
        dummy = torch.zeros(1, seq_len)
        wave_dim = int(haar_multiscale_features(dummy).shape[1])   # now dynamic
        in_dim = seq_len + wave_dim

        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, pred_len),
        )

    def forward(self, x_target):
        wave = haar_multiscale_features(x_target)
        feat = torch.cat([x_target, wave], dim=1)
        return self.net(feat)

class PrototypeResidualBank(nn.Module):
    """
    K residual experts specialized by offline prototypes.
    """
    def __init__(self, seq_len, pred_len, num_prototypes=4, hidden=128):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.num_prototypes = int(num_prototypes)

        dummy = torch.zeros(1, seq_len)
        wave_dim = int(haar_multiscale_features(dummy).shape[1])
        proto_dim = int(build_prototype_features_torch(dummy).shape[1])
        in_dim = seq_len + wave_dim

        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.GELU(),
                nn.Linear(hidden, hidden),
                nn.GELU(),
                nn.Linear(hidden, pred_len),
            )
            for _ in range(self.num_prototypes)
        ])

        self.register_buffer("prototype_centers", torch.zeros(self.num_prototypes, proto_dim))

    def set_prototypes(self, centers_np):
        centers = torch.tensor(centers_np, dtype=torch.float32)
        if centers.shape[0] < self.num_prototypes:
            pad = centers[-1:, :].repeat(self.num_prototypes - centers.shape[0], 1)
            centers = torch.cat([centers, pad], dim=0)
        elif centers.shape[0] > self.num_prototypes:
            centers = centers[:self.num_prototypes]
        self.prototype_centers = centers

    def routing_from_features(self, proto_feat):
        dist = torch.cdist(proto_feat, self.prototype_centers.to(proto_feat.device), p=2) ** 2
        weights = torch.softmax(-dist / PCWRA_CFG["prototype_temperature"], dim=1)
        return weights, dist

    def forward(self, x_target, proto_feat):
        wave = haar_multiscale_features(x_target)
        feat = torch.cat([x_target, wave], dim=1)

        weights, dist = self.routing_from_features(proto_feat)

        expert_outs = []
        for head in self.heads:
            expert_outs.append(head(feat))
        expert_outs = torch.stack(expert_outs, dim=1)  # [B,K,H]

        resid = (weights.unsqueeze(-1) * expert_outs).sum(dim=1)
        return resid, weights, dist, expert_outs

class PCWRA(nn.Module):
    """
    Modes:
    - base_only
    - fixed_global_residual
    - pcwra
    """
    def __init__(self, seq_len, pred_len, target_idx, num_prototypes=4, mode="pcwra"):
        super().__init__()
        assert mode in ["base_only", "fixed_global_residual", "pcwra"]
        self.mode = mode
        self.target_idx = int(target_idx)

        self.base = DLinearTargetOnly(seq_len=seq_len, pred_len=pred_len, kernel_size=25)
        self.global_resid = GlobalResidualMLP(seq_len=seq_len, pred_len=pred_len, hidden=PCWRA_CFG["global_resid_hidden"])
        self.proto_bank = PrototypeResidualBank(
            seq_len=seq_len,
            pred_len=pred_len,
            num_prototypes=num_prototypes,
            hidden=PCWRA_CFG["resid_hidden"],
        )

    def set_prototypes(self, centers_np):
        self.proto_bank.set_prototypes(centers_np)

    def forward(self, x_scaled):
        x_target = x_scaled[:, :, self.target_idx]
        base_pred = self.base(x_target)

        # torch-native prototype features, no CPU roundtrip
        proto_feat = build_prototype_features_torch(x_target)

        if self.mode == "base_only":
            B = x_target.shape[0]
            zeros = torch.zeros_like(base_pred)
            aux = {
                "base_pred": base_pred,
                "resid_pred": zeros,
                "proto_weights": torch.zeros(B, self.proto_bank.num_prototypes, device=x_target.device),
                "proto_entropy": torch.zeros(B, device=x_target.device),
                "proto_feat": proto_feat,
                "expert_outs": None,
            }
            return base_pred, aux

        if self.mode == "fixed_global_residual":
            resid = self.global_resid(x_target)
            B = x_target.shape[0]
            aux = {
                "base_pred": base_pred,
                "resid_pred": resid,
                "proto_weights": torch.zeros(B, self.proto_bank.num_prototypes, device=x_target.device),
                "proto_entropy": torch.zeros(B, device=x_target.device),
                "proto_feat": proto_feat,
                "expert_outs": None,
            }
            return base_pred + resid, aux

        # pcwra
        resid, weights, dist, expert_outs = self.proto_bank(x_target, proto_feat)
        proto_entropy = -(weights.clamp_min(1e-8).log() * weights).sum(dim=1)

        aux = {
            "base_pred": base_pred,
            "resid_pred": resid,
            "proto_weights": weights,
            "proto_entropy": proto_entropy,
            "proto_feat": proto_feat,
            "expert_outs": expert_outs,
        }
        return base_pred + resid, aux

# -------------------------------------------------
# Prototype fitting
# -------------------------------------------------
def fit_prototypes_for_dataset(dataset_name, bundle, seq_len, num_prototypes=4, max_samples=4000, seed=42):
    values = np.asarray(bundle["values_scaled"])
    target_idx = int(bundle["target_idx"])

    train_end = resolve_train_end_from_bundle_or_disk(dataset_name, bundle)
    x = values[:train_end, target_idx]

    windows = []
    max_start = max(1, len(x) - seq_len)
    step = max(1, max_start // max_samples)

    for s in range(0, max_start, step):
        w = x[s:s + seq_len]
        if len(w) == seq_len:
            windows.append(w)
        if len(windows) >= max_samples:
            break

    if len(windows) == 0:
        raise RuntimeError(f"No prototype windows could be built for dataset={dataset_name}")

    windows = np.stack(windows, axis=0).astype(np.float32)
    feats = build_prototype_features_np(windows)

    k = min(num_prototypes, len(feats))
    if k < 2:
        k = 1

    km = KMeans(n_clusters=k, random_state=seed, n_init=10)
    km.fit(feats)
    centers = km.cluster_centers_.astype(np.float32)

    if k < num_prototypes:
        pad = np.repeat(centers[-1:, :], num_prototypes - k, axis=0)
        centers = np.concatenate([centers, pad], axis=0)

    return centers.astype(np.float32), feats

# -------------------------------------------------
# Eval + train
# -------------------------------------------------
@torch.no_grad()
def evaluate_model(model, loader, target_mean, target_std):
    model.eval()
    preds = []
    trues = []
    entropy_rows = []
    weight_rows = []

    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_scaled, aux = model(x)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
        entropy_rows.append(aux["proto_entropy"].detach().cpu().numpy())
        weight_rows.append(aux["proto_weights"].detach().cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    ent = np.concatenate(entropy_rows, axis=0)
    weights = np.concatenate(weight_rows, axis=0)

    avg_weights = weights.mean(axis=0) if weights.size > 0 else np.zeros(PCWRA_CFG["num_prototypes"], dtype=np.float32)

    return {
        "preds": preds,
        "trues": trues,
        "mse": mse(trues.reshape(-1), preds.reshape(-1)),
        "mae": mae(trues.reshape(-1), preds.reshape(-1)),
        "rmse": rmse(trues.reshape(-1), preds.reshape(-1)),
        "avg_proto_entropy": float(ent.mean()) if ent.size > 0 else 0.0,
        "avg_proto_weights": avg_weights,
    }

def train_one_run(dataset_name, mode):
    set_seed(SEED)

    bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
    pred_len = p5.resolve_pred_len(dataset_name, "long")
    seq_len = resolve_seq_len(dataset_name)
    batch_size = resolve_batch_size(dataset_name)
    epochs = resolve_epochs(dataset_name)

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    centers = None
    if mode == "pcwra":
        centers, _ = fit_prototypes_for_dataset(
            dataset_name=dataset_name,
            bundle=bundle,
            seq_len=seq_len,
            num_prototypes=PCWRA_CFG["num_prototypes"],
            max_samples=4000,
            seed=SEED,
        )

    model = PCWRA(
        seq_len=seq_len,
        pred_len=pred_len,
        target_idx=int(target_idx),
        num_prototypes=PCWRA_CFG["num_prototypes"],
        mode=mode,
    ).to(p5.DEVICE)

    if mode == "pcwra":
        model.set_prototypes(centers)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
        foreach=False,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_rmse = float("inf")
    best_epoch = -1
    best_state = None
    patience_left = PATIENCE

    history = []
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()

        loss_vals = []
        point_vals = []
        ps_vals = []
        keep_vals = []
        entropy_vals = []
        resid_aux_vals = []

        for batch in train_loader:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred_scaled, aux = model(x)
                loss, parts = pcwra_loss(pred_scaled, y, aux, mode)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_vals.append(float(loss.detach().cpu().item()))
            point_vals.append(parts["point_mse"])
            ps_vals.append(parts["ps_loss"])
            keep_vals.append(parts["selective_keep"])
            entropy_vals.append(parts["entropy_reg"])
            resid_aux_vals.append(parts["resid_aux"])

        val_eval = evaluate_model(model, val_loader, target_mean, target_std)
        val_rmse = float(val_eval["rmse"])

        history.append({
            "dataset": dataset_name,
            "mode": mode,
            "epoch": epoch,
            "train_loss": float(np.mean(loss_vals)),
            "train_point_mse": float(np.mean(point_vals)),
            "train_ps_loss": float(np.mean(ps_vals)),
            "train_selective_keep": float(np.mean(keep_vals)),
            "train_entropy_reg": float(np.mean(entropy_vals)),
            "train_resid_aux": float(np.mean(resid_aux_vals)),
            "val_rmse": val_rmse,
            "val_mae": float(val_eval["mae"]),
            "val_mse": float(val_eval["mse"]),
            "val_avg_proto_entropy": float(val_eval["avg_proto_entropy"]),
            "val_avg_proto_weights_json": json.dumps(val_eval["avg_proto_weights"].tolist()),
        })

        print(
            f"[{dataset_name} | {mode}] epoch {epoch:03d} | "
            f"train_loss={np.mean(loss_vals):.6f} | val_rmse={val_rmse:.6f} | "
            f"proto_entropy={val_eval['avg_proto_entropy']:.4f} | batch={batch_size}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{dataset_name} | {mode}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    test_eval = evaluate_model(model, test_loader, target_mean, target_std)

    ckpt_path = OUT_CKPTS / f"{dataset_name}_{mode}.pt"
    pred_path = OUT_PREDS / f"{dataset_name}_{mode}_test_predictions.npz"

    save_obj = {
        "dataset": dataset_name,
        "mode": mode,
        "pcwra_cfg": PCWRA_CFG,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }
    if centers is not None:
        save_obj["prototype_centers"] = centers
    torch.save(save_obj, ckpt_path)

    np.savez_compressed(
        pred_path,
        preds=test_eval["preds"],
        trues=test_eval["trues"],
        seq_len=seq_len,
        pred_len=pred_len,
        avg_proto_entropy=np.array([test_eval["avg_proto_entropy"]], dtype=np.float32),
        avg_proto_weights=np.array(test_eval["avg_proto_weights"], dtype=np.float32),
    )

    row = {
        "dataset": dataset_name,
        "mode": mode,
        "model": f"PCWRA_{mode}",
        "family": "wavelet_prototype_residual",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "batch_size": batch_size,
        "epochs_target": epochs,
        "best_epoch": int(best_epoch),
        "train_seconds": float(time.time() - t0),
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(test_eval["mse"]),
        "test_mae": float(test_eval["mae"]),
        "test_rmse": float(test_eval["rmse"]),
        "avg_proto_entropy": float(test_eval["avg_proto_entropy"]),
        "avg_proto_weights_json": json.dumps(test_eval["avg_proto_weights"].tolist()),
    }

    hist_df = pd.DataFrame(history)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, hist_df

In [10]:
# ============================================
# PILOT RUN
# ============================================

pilot_rows = []
pilot_hist = []

if RUN_MODE in ["pilot_only", "pilot_then_full"]:
    print("Starting pilot...")
    for ds in PILOT_DATASETS:
        for mode in ["base_only", "fixed_global_residual", "pcwra"]:
            print("\n" + "=" * 120)
            print(f"PILOT | dataset={ds} | mode={mode}")
            print("=" * 120)
            row, hist_df = train_one_run(ds, mode)
            pilot_rows.append(row)
            pilot_hist.append(hist_df)

    pilot_df = pd.DataFrame(pilot_rows)
    pilot_hist_df = pd.concat(pilot_hist, ignore_index=True)

    pilot_metrics_csv = OUT_TABLES / "pilot_metrics.csv"
    pilot_history_csv = OUT_TABLES / "pilot_history.csv"
    pilot_df.to_csv(pilot_metrics_csv, index=False)
    pilot_hist_df.to_csv(pilot_history_csv, index=False)

    print("\nSaved:", pilot_metrics_csv)
    print("Saved:", pilot_history_csv)
    display(pilot_df)

    pivot = pilot_df.pivot(index="dataset", columns="mode", values="test_rmse").reset_index()
    pivot["pcwra_vs_base_gain"] = pivot["base_only"] - pivot["pcwra"]
    pivot["pcwra_vs_global_gain"] = pivot["fixed_global_residual"] - pivot["pcwra"]

    base_wins = int((pivot["pcwra_vs_base_gain"] > 0).sum())
    global_wins = int((pivot["pcwra_vs_global_gain"] > 0).sum())

    pilot_summary = pivot.copy()
    pilot_summary["pilot_pass_base"] = base_wins
    pilot_summary["pilot_pass_global"] = global_wins

    pilot_summary_csv = OUT_TABLES / "pilot_summary.csv"
    pilot_summary.to_csv(pilot_summary_csv, index=False)

    print("\nSaved:", pilot_summary_csv)
    display(pilot_summary)

    PILOT_GOOD = (base_wins >= PILOT_THRESH_BASE) and (global_wins >= PILOT_THRESH_GLOBAL)
    print("\nPILOT DECISION")
    print(f"pcwra beats base_only on {base_wins}/{len(PILOT_DATASETS)} pilot datasets")
    print(f"pcwra beats fixed_global_residual on {global_wins}/{len(PILOT_DATASETS)} pilot datasets")
    print("PILOT_GOOD =", PILOT_GOOD)
else:
    PILOT_GOOD = False

Starting pilot...

PILOT | dataset=etth1 | mode=base_only


/tmp/ipykernel_3106841/2424999958.py:242: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_3106841/2424999958.py:268: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


[etth1 | base_only] epoch 001 | train_loss=0.603788 | val_rmse=3.494762 | proto_entropy=0.0000 | batch=512


/tmp/ipykernel_3106841/2424999958.py:166: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


[etth1 | base_only] epoch 002 | train_loss=0.237841 | val_rmse=3.241314 | proto_entropy=0.0000 | batch=512
[etth1 | base_only] epoch 003 | train_loss=0.205117 | val_rmse=3.123627 | proto_entropy=0.0000 | batch=512
[etth1 | base_only] epoch 004 | train_loss=0.191193 | val_rmse=3.057716 | proto_entropy=0.0000 | batch=512
[etth1 | base_only] epoch 005 | train_loss=0.184180 | val_rmse=3.013762 | proto_entropy=0.0000 | batch=512
[etth1 | base_only] epoch 006 | train_loss=0.179150 | val_rmse=2.982876 | proto_entropy=0.0000 | batch=512
[etth1 | base_only] epoch 007 | train_loss=0.175747 | val_rmse=2.958461 | proto_entropy=0.0000 | batch=512
[etth1 | base_only] epoch 008 | train_loss=0.173099 | val_rmse=2.936506 | proto_entropy=0.0000 | batch=512
[etth1 | base_only] epoch 009 | train_loss=0.171313 | val_rmse=2.925543 | proto_entropy=0.0000 | batch=512
[etth1 | base_only] epoch 010 | train_loss=0.168704 | val_rmse=2.909844 | proto_entropy=0.0000 | batch=512
[etth1 | base_only] epoch 011 | train

RuntimeError: mat1 and mat2 shapes cannot be multiplied (512x124 and 120x128)

In [ ]:
# ============================================
# FULL RUN (pcwra only) IF PILOT IS GOOD
# ============================================

full_rows = []
full_hist = []

RUN_FULL = (
    (RUN_MODE == "full_only") or
    (RUN_MODE == "pilot_then_full" and AUTO_FULL_IF_PILOT_GOOD and PILOT_GOOD)
)

print("RUN_FULL =", RUN_FULL)

if RUN_FULL:
    for ds in ALL_DATASETS:
        print("\n" + "=" * 120)
        print(f"FULL RUN | dataset={ds} | mode=pcwra")
        print("=" * 120)
        row, hist_df = train_one_run(ds, "pcwra")
        full_rows.append(row)
        full_hist.append(hist_df)

    full_df = pd.DataFrame(full_rows)
    full_hist_df = pd.concat(full_hist, ignore_index=True)

    full_metrics_csv = OUT_TABLES / "full_metrics.csv"
    full_history_csv = OUT_TABLES / "full_history.csv"
    full_df.to_csv(full_metrics_csv, index=False)
    full_hist_df.to_csv(full_history_csv, index=False)

    print("\nSaved:", full_metrics_csv)
    print("Saved:", full_history_csv)
    display(full_df)
else:
    print("Full run skipped. Stop here if pilot failed.")

In [ ]:
# ============================================
# MERGE WITH CURRENT MASTER AND COMPARE AGAINST CURRENT BEST
# ============================================

if RUN_FULL:
    if not MASTER_PATH.exists():
        raise FileNotFoundError(f"Missing master file: {MASTER_PATH}")
    if not BEST_PATH.exists():
        raise FileNotFoundError(f"Missing best file: {BEST_PATH}")

    master = pd.read_csv(MASTER_PATH)
    best_old = pd.read_csv(BEST_PATH)

    master_candidate = master[master["model"] != "PCWRA_pcwra"].copy()

    new_rows = []
    for _, r in full_df.iterrows():
        new_rows.append(
            metric_row_from_npz(
                dataset=r["dataset"],
                model=r["model"],
                family=r["family"],
                seq_len=r["seq_len"],
                pred_len=r["pred_len"],
                pred_file_rel=r["prediction_file"],
            )
        )

    new_df = pd.DataFrame(new_rows)
    master_candidate = pd.concat([master_candidate, new_df], ignore_index=True)
    master_candidate = master_candidate.sort_values(["dataset", "family", "model"]).reset_index(drop=True)

    best_candidate = (
        master_candidate.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
                        .groupby("dataset", as_index=False)
                        .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
    )

    candidate_master_csv = ROOT / "results" / "tables" / "master_all_models_metrics_unified_pcwra_candidate.csv"
    candidate_best_csv = ROOT / "results" / "tables" / "master_best_by_dataset_unified_pcwra_candidate.csv"

    master_candidate.to_csv(candidate_master_csv, index=False)
    best_candidate.to_csv(candidate_best_csv, index=False)

    compare = best_old.rename(columns={
        "model": "old_best_model",
        "family": "old_best_family",
        "test_mse": "old_best_mse",
        "test_mae": "old_best_mae",
        "test_rmse": "old_best_rmse",
    }).merge(
        full_df[["dataset", "model", "test_mse", "test_mae", "test_rmse", "avg_proto_entropy", "avg_proto_weights_json"]].rename(columns={
            "model": "pcwra_model",
            "test_mse": "pcwra_mse",
            "test_mae": "pcwra_mae",
            "test_rmse": "pcwra_rmse",
        }),
        on="dataset",
        how="left"
    )

    compare["rmse_gain_abs"] = compare["old_best_rmse"] - compare["pcwra_rmse"]
    compare["rmse_gain_pct"] = 100.0 * compare["rmse_gain_abs"] / compare["old_best_rmse"]

    compare["mae_gain_abs"] = compare["old_best_mae"] - compare["pcwra_mae"]
    compare["mae_gain_pct"] = 100.0 * compare["mae_gain_abs"] / compare["old_best_mae"]

    compare_csv = OUT_TABLES / "full_vs_current_best.csv"
    compare.to_csv(compare_csv, index=False)

    wins = int((compare["rmse_gain_abs"] > 0).sum())
    losses = int((compare["rmse_gain_abs"] < 0).sum())
    ties = int((compare["rmse_gain_abs"] == 0).sum())

    print("\nSaved:", candidate_master_csv)
    print("Saved:", candidate_best_csv)
    print("Saved:", compare_csv)
    display(compare)
    display(best_candidate)

    print("\nFINAL DECISION")
    print(f"PCWRA wins on {wins}/{len(ALL_DATASETS)} datasets by RMSE")
    print(f"Loses on {losses}/{len(ALL_DATASETS)} datasets")
    print(f"Ties on {ties}/{len(ALL_DATASETS)} datasets")

    if wins >= 4:
        print("RECOMMENDATION: promising enough to keep and analyze further.")
    else:
        print("RECOMMENDATION: not strong enough overall; reject or redesign.")
else:
    print("No full-run merge because RUN_FULL is False.")

## Exact module changes

1. Replace the previous monolithic adaptive wavelet forecaster with:
   - `DLinearTargetOnly`
   - `GlobalResidualMLP`
   - `PrototypeResidualBank`
   - `PCWRA`

2. Add offline prototype fitting per dataset:
   - extract training-window target-channel features
   - fit `KMeans`
   - store centers in the model

3. Use prototype-conditioned residual correction instead of a learned online router.

## Exact pseudocode

1. Fit dataset-specific prototypes from training windows.
2. For each batch:
   - base forecast from simple target-channel expert
   - compute prototype feature for each sample
   - soft-assign sample to prototypes
   - get weighted residual from prototype heads
   - add residual to base
   - train with point MSE + patch structural loss + selective weighting + residual auxiliary loss

## Forward-pass equations

Base:
\[
\hat y_b = f_b(x^{(t)})
\]

Prototype features:
\[
z = \phi(x^{(t)})
\]

Soft assignment:
\[
a_k = \frac{\exp(-\|z-c_k\|^2 / \tau)}{\sum_j \exp(-\|z-c_j\|^2 / \tau)}
\]

Prototype residuals:
\[
r_k = h_k([x^{(t)}, \psi(x^{(t)})])
\]
where \(\psi\) is a fixed Haar multiscale feature map.

Final prediction:
\[
\hat y = \hat y_b + \sum_{k=1}^{K} a_k r_k
\]

## Loss equations

Residual target:
\[
r^* = y - \hat y_b
\]

Point-wise selective loss:
\[
\mathcal{L}_{sel} = \frac{\sum_i m_i \cdot \text{MSE}(\hat y_i, y_i)}{\sum_i m_i + \epsilon}
\]
where \(m_i\) keeps low-entropy routing samples.

Patch structural loss:
\[
\mathcal{L}_{ps} = \text{MSE}(\mu(\hat y), \mu(y))
+ \text{MSE}(\sigma^2(\hat y), \sigma^2(y))
+ \mathbb{E}(1-\rho(\hat y, y))^2
\]

Residual auxiliary loss:
\[
\mathcal{L}_{res} = \text{MSE}(\hat r, r^*)
\]

Total:
\[
\mathcal{L} =
\lambda_{sel}\mathcal{L}_{sel}
+ \lambda_{ps}\mathcal{L}_{ps}
+ \lambda_{res}\mathcal{L}_{res}
+ \lambda_{ent} \mathbb{E}[H(a)]
\]

## Exact ablations

Pilot/full comparisons:
- `base_only`
- `fixed_global_residual`
- `pcwra`

Then ablate:
- remove structural loss
- remove selective weighting
- change number of prototypes
- replace Haar features with raw-only residual heads
- hard nearest-prototype vs soft routing

## Cheapest pilot first

Run only:
- ETTh1
- Weather
- Traffic
- ILI

Continue only if:
- `pcwra` beats `base_only` on at least 3/4
- `pcwra` beats `fixed_global_residual` on at least 3/4

## Final 9-dataset schedule

Only if pilot passes, run:
ETTh1, ETTh2, ETTm1, ETTm2, Weather, Electricity, Traffic, Exchange, ILI

## Failure signals to monitor

Reject if:
- pilot fails the 3/4 and 3/4 rule
- prototype entropy stays extremely high everywhere
- one prototype dominates every dataset
- full run improves too few datasets

## Expected metrics table layout

- `dataset`
- `mode`
- `model`
- `family`
- `seq_len`
- `pred_len`
- `batch_size`
- `best_epoch`
- `test_mse`
- `test_mae`
- `test_rmse`
- `avg_proto_entropy`
- `avg_proto_weights_json`

## Result threshold that justifies continuing

Continue only if:
- pilot passes, and
- full run beats current best on at least 4/9 datasets
or clearly improves hard regimes with interpretable prototype specialization.